# Cloud-SOAR-Triage — Full 17×5-seed run on Colab (GPU)

Runs `scripts/multiseed_runner.py` for all 17 configurations × 5 seeds and
reports mean ± std (paper §IV). **Resumable**: results are saved to Google
Drive after every run, so if Colab disconnects just re-run cells 1–4.

Cell 4 lets you choose the benchmark: **`temporal`** (chronological real-flow
split) or **`iid`** (the controlled i.i.d. benchmark). Each writes a separate
results file, so you can run both.

### Before you start
1. Runtime → **Change runtime type** → Hardware accelerator = **T4 GPU**.
2. Upload **`cloud-soar-triage_colab.zip`** to **My Drive** (root). Do NOT unzip it.

In [ ]:
# 1) Mount Google Drive (results are saved here so they survive disconnects)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Extract the project to LOCAL Colab storage (fresh each run). Finds
#    train.py at ANY depth, and prints diagnostics if anything is off.
import os, glob, zipfile, shutil

LOCAL = '/content/cloud-soar-triage'
zips = sorted(glob.glob('/content/drive/MyDrive/**/cloud-soar-triage*.zip', recursive=True))
print('Zip files found on Drive:', zips)
assert zips, 'No zip on Drive. Upload cloud-soar-triage_colab.zip to My Drive, then re-run.'

if os.path.isdir(LOCAL):
    shutil.rmtree(LOCAL)
os.makedirs(LOCAL, exist_ok=True)
with zipfile.ZipFile(zips[0]) as z:
    z.extractall(LOCAL)

train_dirs = [dp for dp, _, fs in os.walk(LOCAL) if 'train.py' in fs]
print('Directories containing train.py:', train_dirs)
if not train_dirs:
    print('Top-level of extracted folder:', os.listdir(LOCAL))
assert train_dirs, 'train.py not found after extract -- the zip may be incomplete; re-upload it.'

PROJECT = next((d for d in train_dirs
                if os.path.isfile(os.path.join(d, 'data', 'processed', 'benchmark_train.pkl'))),
               train_dirs[0])
os.chdir(PROJECT)
print('Project:', PROJECT)
for d in ('data/processed', 'data/processed_temporal'):
    print(f'  {d}/benchmark_train.pkl present:', os.path.isfile(d + '/benchmark_train.pkl'))

In [ ]:
# 3) Verify GPU (torch ships with Colab). Install the few extra deps.
!pip -q install xgboost scikit-learn pyyaml 2>/dev/null
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > T4 GPU, then re-run.'

In [ ]:
# 4) RUN (resumable). Re-run this cell after any disconnect to continue.
#    Pick the benchmark: 'temporal' (chronological real-flow split) or 'iid'.
BENCHMARK = 'temporal'          # <-- change to 'iid' for the i.i.d. benchmark
DATA = 'data/processed_temporal' if BENCHMARK == 'temporal' else 'data/processed'
CKPT = f'/content/ms_ckpts_{BENCHMARK}'
OUT  = f'/content/drive/MyDrive/cloud-soar-triage_results/multiseed_{BENCHMARK}.json'
os.makedirs(os.path.dirname(OUT), exist_ok=True)
print('Benchmark:', BENCHMARK, '| data:', DATA, '| out:', OUT)
!python scripts/multiseed_runner.py \
    --device cuda --seeds 42 43 44 45 46 \
    --max_epochs 100 --patience 10 \
    --processed_dir "$DATA" --checkpoint_dir "$CKPT" --out "$OUT"

In [ ]:
# 5) Inspect progress / final mean ± std for the benchmark chosen above
import json
r = json.load(open(OUT))
print('Benchmark:', BENCHMARK)
print('Seeds completed per config:')
for v, reps in r['raw'].items():
    if reps:
        print(f"  {v:14}: {sorted(int(x['seed']) for x in reps)}")
print()
print(f"{'Config':16}{'W-F1 (mean +/- std)':>26}")
for v, agg in r['summary'].items():
    wf = agg.get('weighted_f1')
    if wf:
        print(f"  {v:14}{wf['mean']:.4f} +/- {wf['std']:.4f}  (n={wf['n']})")